# 📊 Visualisation
**Requires:** an Excel file in the `data/` folder (generated by one of the three data notebooks)

This notebook reads the Excel file you downloaded and produces charts:
- **Price Evolution** — line chart of prices or levels over time
- **Return Distribution** — histogram of period returns
- **Correlation Heatmap** — how closely the series move together

### How to use
1. Run the Setup cell
2. Select the Excel file and sheet
3. Choose the series (tickers/columns) to plot
4. Run the chart cells


In [ ]:
# ── Setup — Run this cell first ─────────────────────────────────────────────
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("..") / "data"

print("✅ Setup complete.")


---
## Step 1 — Load Data


In [ ]:
# Find available Excel files in data/
excel_files = sorted(DATA_DIR.glob("*.xlsx"))

if not excel_files:
    print("⚠️  No Excel files found in data/. Run one of the data notebooks first.")
else:
    file_dropdown = widgets.Dropdown(
        options=[f.name for f in excel_files],
        description="File:",
        style={"description_width": "50px"},
        layout=widgets.Layout(width="350px"),
    )
    load_btn = widgets.Button(description="Load File", button_style="primary")
    out_load = widgets.Output()

    display(file_dropdown, load_btn, out_load)

    _loaded = {}

    def on_load(b):
        with out_load:
            clear_output(wait=True)
            fpath = DATA_DIR / file_dropdown.value
            try:
                xl = pd.ExcelFile(fpath)
                _loaded["sheets"] = xl.sheet_names
                _loaded["path"] = fpath
                print(f"✅ Loaded: {file_dropdown.value}")
                print(f"   Sheets: {', '.join(xl.sheet_names)}")
            except Exception as e:
                print(f"❌ Error loading file: {e}")

    load_btn.on_click(on_load)


---
## Step 2 — Select Sheet & Columns


In [ ]:
# Run this cell after loading the file above

if not _loaded:
    print("⚠️  Load a file first (Step 1).")
else:
    sheet_dd = widgets.Dropdown(
        options=_loaded.get("sheets", []),
        description="Sheet:",
        style={"description_width": "60px"},
        layout=widgets.Layout(width="300px"),
    )
    select_btn = widgets.Button(description="Select Sheet", button_style="primary")
    out_sheet = widgets.Output()
    col_select = widgets.SelectMultiple(
        options=[],
        description="Columns:",
        layout=widgets.Layout(width="400px", height="150px"),
        style={"description_width": "70px"},
    )

    display(sheet_dd, select_btn, out_sheet, col_select)

    _data = {}

    def on_select(b):
        with out_sheet:
            clear_output(wait=True)
            try:
                df = pd.read_excel(_loaded["path"], sheet_name=sheet_dd.value, index_col=0)
                # Keep only numeric columns for plotting
                df = df.select_dtypes(include="number")
                df.index = pd.to_datetime(df.index, errors="coerce")
                _data["df"] = df
                col_select.options = list(df.columns)
                col_select.value = list(df.columns[:min(5, len(df.columns))])
                print(f"✅ Sheet loaded — {len(df)} rows × {len(df.columns)} columns")
            except Exception as e:
                print(f"❌ {e}")

    select_btn.on_click(on_select)


---
## Chart 1 — Price / Level Evolution


In [ ]:
if not _data:
    print("⚠️  Select a sheet first (Step 2).")
else:
    df = _data["df"][list(col_select.value)].dropna(how="all")

    fig, ax = plt.subplots(figsize=(12, 5))
    for col in df.columns:
        ax.plot(df.index, df[col], label=col, linewidth=1.6)

    ax.set_title(f"{sheet_dd.value} — Price / Level", fontsize=14, fontweight="bold")
    ax.set_xlabel("Date")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.2f}"))
    ax.legend(loc="upper left", fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


---
## Chart 2 — Return Distribution


In [ ]:
if not _data:
    print("⚠️  Select a sheet first (Step 2).")
else:
    df = _data["df"][list(col_select.value)].dropna(how="all")
    returns = df.pct_change().dropna(how="all")

    n = len(returns.columns)
    cols_grid = min(n, 3)
    rows_grid = (n + cols_grid - 1) // cols_grid

    fig, axes = plt.subplots(rows_grid, cols_grid,
                              figsize=(5 * cols_grid, 4 * rows_grid),
                              squeeze=False)

    for i, col in enumerate(returns.columns):
        ax = axes[i // cols_grid][i % cols_grid]
        sns.histplot(returns[col].dropna(), bins=40, kde=True, ax=ax,
                     color=sns.color_palette("tab10")[i % 10])
        ax.set_title(col, fontsize=11, fontweight="bold")
        ax.set_xlabel("Return")
        ax.set_ylabel("Count")
        ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
        ax.spines[["top", "right"]].set_visible(False)

    # Hide empty subplots
    for j in range(i + 1, rows_grid * cols_grid):
        axes[j // cols_grid][j % cols_grid].set_visible(False)

    fig.suptitle("Return Distributions", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()


---
## Chart 3 — Correlation Heatmap


In [ ]:
if not _data:
    print("⚠️  Select a sheet first (Step 2).")
else:
    df = _data["df"][list(col_select.value)].dropna(how="all")
    returns = df.pct_change().dropna(how="all")
    corr = returns.corr()

    size = max(6, len(corr) * 0.9)
    fig, ax = plt.subplots(figsize=(size, size * 0.85))

    mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)
    import numpy as np
    mask.values[np.triu_indices_from(mask, k=1)] = True

    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=-1, vmax=1,
        linewidths=0.5,
        ax=ax,
        annot_kws={"size": 9},
    )
    ax.set_title("Return Correlation Heatmap", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
